# Kernel Density Estimation

The samples in the advertising data come from a *continuous* random variable, so their distribution is described by a PDF. Can we recover an idea of that PDF from the data? A histogram is a first attempt — technically it *is* a density (positive, area 1) — but its blocky bars and artificial peaks come from the binning, not the data. **Kernel density estimation (KDE)** does better: place a little Gaussian "mountain" on every data point, sum them, divide by $n$ — a smooth, valid density estimate. It is based on the fourteenth video of Lesson 3 (Describing Distributions).

## 1. Introduction

You know how to summarize data with numbers (moments, quantiles) and pictures (box plots). The most ambitious summary is the distribution itself: an estimate of the PDF the data came from. That estimate powers the smooth density curves you see layered over histograms, the violin plots that fix box plots' blind spots, and countless "distribution of X" figures in data science.

After completing this notebook you will be able to:

- Explain why a histogram is a legitimate but crude density estimate.
- Build a **kernel density estimate**: a Gaussian kernel on each observation, averaged.
- Show that the KDE integrates to 1 (it is a genuine density).
- Use the **bandwidth** $\sigma$ to trade off smoothness vs. detail, and explain what more data buys you.

## 2. Theory

### Histograms as density estimates — and their flaws

Normalize a histogram (divide the counts by $n \cdot \text{bin width}$) and it satisfies all the conditions of a density function: it is positive, and the area under it is 1. But it is not a great approximation of the underlying PDF:

1. PDFs are usually **smooth** functions; a histogram is a staircase.
2. The **discontinuities and peaks come from the technique** — the choice of bins — not from the data itself. The true distribution behind the advertising data may be perfectly smooth even when its histogram looks jagged.

### The KDE recipe

Kernel density estimation replaces each bar with something gentler:

1. **Mark the observations** $x_1, \dots, x_n$ on the axis.
2. Give each point an effect that **spreads around it** — we want high density where points cluster and low density where there are none. Concretely, put a *little mountain* on each point: a small Gaussian curve centered there. This function is called the **kernel**. (Other kernel shapes work too; we stick with the Gaussian.)
3. The $\sigma$ of the Gaussians — the **bandwidth** — determines how far each point's effect spreads: thin mountains for small $\sigma$, wide ones for large $\sigma$.
4. **Sum all the curves and multiply by $\tfrac1n$:**

$$
\hat f(x) = \frac{1}{n} \sum_{i=1}^{n} \frac{1}{\sigma\sqrt{2\pi}} \, e^{-\frac{(x - x_i)^2}{2\sigma^2}}
$$

### Why it is a genuine density

Each Gaussian has area 1 under it, so the *average* of $n$ of them also has area exactly 1 — and the sum of positive functions is positive. The KDE is a legitimate PDF by construction.

### What to expect

From just 12 points the estimate looks lumpy — you are asking a lot of 12 numbers. But from many points the sum of mountains melts into a really nice smooth curve that resembles the true PDF well. KDE is *the* standard way to approximate a density from data.

## 3. Worked Examples

### Example 1: a two-point KDE by hand

Data: $x_1 = 0$, $x_2 = 4$, bandwidth $\sigma = 1$. The estimate is

$$
\hat f(x) = \tfrac12\big[\varphi(x; 0, 1) + \varphi(x; 4, 1)\big]
$$

where $\varphi$ is the normal PDF. At $x = 0$: $\hat f(0) = \tfrac12[0.3989 + 0.0001] \approx 0.2$. At the midpoint $x = 2$: $\hat f(2) = \tfrac12[0.054 + 0.054] = 0.054$ — a valley between two mountains, as it should be.

### Example 2: bandwidth extremes

- $\sigma \to 0$: each mountain collapses to a spike at its data point — the estimate memorizes the sample (undersmoothing).
- $\sigma$ huge: every mountain covers everything — the estimate becomes one broad blob that ignores the structure (oversmoothing).

Good bandwidths sit in between; a common default (Silverman's rule of thumb) is $\sigma \approx 1.06\, s\, n^{-1/5}$ with $s$ the sample standard deviation.

### Example 3: KDE vs. histogram on the newspaper sample

With the 12 budgets and a moderate bandwidth, the KDE shows one main bump around \$20–30k with a gentle shoulder toward the large budgets — the same story as the box plot's right skew, but as a smooth curve rather than bars or a box.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

newspaper = np.array([8.7, 14.2, 18.3, 18.4, 23.2, 25.9,
                      29.7, 35.2, 51.2, 54.7, 65.9, 75.0])

def gaussian(x, center, sigma):
    return np.exp(-0.5 * ((x - center) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

def kde(x, data, sigma):
    return np.mean([gaussian(x, xi, sigma) for xi in data], axis=0)

### The KDE is a genuine density: area = 1

Whatever the data and bandwidth, the average of unit-area Gaussians integrates to 1.

In [ ]:
# a grid wide enough to hold even the widest mountains' tails
grid = np.linspace(-150, 250, 8001)
for sigma in [2, 5, 10, 25]:
    estimate = kde(grid, newspaper, sigma)
    area = np.trapezoid(estimate, grid)
    print(f"bandwidth sigma = {sigma:>4}: area under the KDE = {area:.6f}")

### The two-point example from the worked examples

In [ ]:
x = np.linspace(-4, 8, 500)
two_points = np.array([0.0, 4.0])
estimate = kde(x, two_points, sigma=1.0)

print(f"f_hat(0) = {kde(np.array([0.0]), two_points, 1.0)[0]:.4f}   (~0.2)")
print(f"f_hat(2) = {kde(np.array([2.0]), two_points, 1.0)[0]:.4f}   (the valley)")
print(f"f_hat(4) = {kde(np.array([4.0]), two_points, 1.0)[0]:.4f}   (~0.2 again)")

### More data means a better estimate

Sample increasing amounts of data from a known distribution and compare the KDE with the true PDF — the mountains melt into the real curve.

In [ ]:
rng = np.random.default_rng(42)

true_x = np.linspace(-1, 11, 500)
true_pdf = 0.6 * gaussian(true_x, 3, 1) + 0.4 * gaussian(true_x, 7, 1.2)

def sample_mixture(n):
    which = rng.random(n) < 0.6
    return np.where(which, rng.normal(3, 1, n), rng.normal(7, 1.2, n))

for n in [12, 100, 10_000]:
    data = sample_mixture(n)
    sigma = 1.06 * data.std() * n**(-1/5)          # Silverman's rule of thumb
    estimate = kde(true_x, data, sigma)
    max_error = np.max(np.abs(estimate - true_pdf))
    print(f"n = {n:>6}: bandwidth = {sigma:.3f}, max |KDE - true PDF| = {max_error:.4f}")

## 5. Visualizations

### The video's build-up: points → mountains → smooth sum

Left: each of the 12 newspaper budgets wears a little Gaussian (each scaled by $\tfrac1n$). Right: their sum is the KDE.

In [ ]:
x = np.linspace(-20, 110, 1000)
sigma = 6
n = len(newspaper)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

for xi in newspaper:
    axes[0].plot(x, gaussian(x, xi, sigma) / n, color="tab:blue", alpha=0.6)
axes[0].plot(newspaper, np.zeros_like(newspaper), "o", color="black",
             label="the 12 data points")
axes[0].set_title("a little Gaussian mountain on every point (each with area 1/n)")
axes[0].legend()

axes[1].plot(x, kde(x, newspaper, sigma), color="tab:red", linewidth=2.5,
             label="KDE = sum of the mountains")
axes[1].plot(newspaper, np.zeros_like(newspaper), "o", color="black")
axes[1].fill_between(x, kde(x, newspaper, sigma), color="tab:red", alpha=0.15)
axes[1].set_title("summing them gives a smooth density (area = 1)")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("newspaper ad budget ($1000s)")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("density")
plt.suptitle("Kernel density estimation, step by step")
plt.tight_layout()
plt.show()

### Histogram vs. KDE

The normalized histogram is a valid density but a jagged one; the KDE tells the same story smoothly.

In [ ]:
x = np.linspace(-20, 110, 1000)

plt.figure(figsize=(10, 5))
plt.hist(newspaper, bins=8, density=True, color="tab:blue", alpha=0.4,
         edgecolor="black", label="histogram (area 1, but blocky)")
plt.plot(x, kde(x, newspaper, 6), color="tab:red", linewidth=2.5,
         label="KDE (smooth, area 1)")
plt.plot(newspaper, np.full_like(newspaper, -0.0005), "|", markersize=15,
         color="black", label="data points")
plt.title("Two density estimates of the same 12 budgets")
plt.xlabel("newspaper ad budget ($1000s)")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The bandwidth dial: undersmoothing to oversmoothing

In [ ]:
x = np.linspace(-30, 120, 1000)

plt.figure(figsize=(10, 5))
for sigma, color in [(1.5, "tab:purple"), (6, "tab:red"), (25, "tab:green")]:
    plt.plot(x, kde(x, newspaper, sigma), color=color, linewidth=2,
             label=f"sigma = {sigma}")
plt.plot(newspaper, np.zeros_like(newspaper), "|", markersize=15, color="black")
plt.title("Small sigma memorizes the points; large sigma blurs everything away")
plt.xlabel("newspaper ad budget ($1000s)")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### From 12 points to 10,000: the estimate converges to the truth

In [ ]:
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, n in zip(axes, [12, 100, 10_000]):
    data = sample_mixture(n)
    sigma = 1.06 * data.std() * n**(-1/5)
    ax.plot(true_x, true_pdf, color="black", linewidth=2, label="true PDF")
    ax.plot(true_x, kde(true_x, data, sigma), color="tab:red", linewidth=2,
            label="KDE")
    ax.set_title(f"n = {n}")
    ax.set_xlabel("value")
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("density")
plt.suptitle("With enough data, the sum of mountains becomes the real curve")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Compute the KDE of the tiny dataset $[2, 3, 7]$ with $\sigma = 1$ at the points $x = 2.5$ and $x = 5$, by hand from the formula (three Gaussian evaluations each), then check with the `kde` function.

**Exercise 2.** Verify numerically that the KDE of Exercise 1 integrates to 1, and that this stays true if you change $\sigma$ to 0.3 or 3.

### Intermediate

**Exercise 3.** The KDE inherits smoothness from its kernel — but not all summaries change. Compute the *mean* of the KDE density (i.e. $\int x \hat f(x)\,dx$) for the newspaper data at several bandwidths and compare with the sample mean. What stays fixed, and can you see why? (Think: each mountain is centered on its data point.)

**Exercise 4.** KDE fixes the box plot's blindness: build the bimodal dataset from the box-plot notebook (normals at 30 and 70) and plot its histogram, box plot, and KDE side by side. Which of the three reveal the two clusters?

### Advanced

**Exercise 5.** The KDE's variance is *inflated*: show numerically that $\int (x - \bar x)^2 \hat f(x) dx = s^2 + \sigma^2$ (sample variance plus bandwidth squared) for the newspaper data. Interpret: each data point was smeared by a $\sigma$-wide Gaussian, and independent spreads add — the sum-of-Gaussians rule from earlier in this lesson.

**Exercise 6.** Sampling from a KDE is easy: pick a random data point, then add Gaussian noise with std $\sigma$. Generate 100,000 samples from the newspaper KDE ($\sigma = 6$) this way, histogram them, and overlay the KDE curve. Why does this two-step recipe produce exactly the KDE density?

## 7. Solutions

In [ ]:
# Exercise 1: KDE by hand at two points
tiny = np.array([2.0, 3.0, 7.0])
for x0 in [2.5, 5.0]:
    parts = [gaussian(x0, xi, 1.0) for xi in tiny]
    by_hand = np.mean(parts)
    print(f"x = {x0}: contributions {np.round(parts, 4)} -> f_hat = {by_hand:.4f}"
          f"   (function: {kde(np.array([x0]), tiny, 1.0)[0]:.4f})")

In [ ]:
# Exercise 2: area is 1 for any bandwidth
grid = np.linspace(-15, 25, 8001)
for s in [0.3, 1.0, 3.0]:
    area = np.trapezoid(kde(grid, tiny, s), grid)
    print(f"Exercise 2: sigma = {s}: area = {area:.6f}")

In [ ]:
# Exercise 3: the KDE's mean equals the sample mean
grid = np.linspace(-150, 250, 20001)
print(f"Exercise 3: sample mean = {newspaper.mean():.4f}")
for s in [2, 6, 25]:
    est = kde(grid, newspaper, s)
    kde_mean = np.trapezoid(grid * est, grid)
    print(f"  sigma = {s:>2}: mean of KDE density = {kde_mean:.4f}")
print("Each mountain is a Gaussian CENTERED at its data point, so it contributes")
print("exactly x_i to the mean; averaging gives the sample mean, any bandwidth.")

In [ ]:
# Exercise 4: KDE reveals what the box plot hides
rng = np.random.default_rng(0)
bimodal = np.concatenate([rng.normal(30, 4, size=500), rng.normal(70, 4, size=500)])
grid = np.linspace(0, 100, 800)
sigma_b = 1.06 * bimodal.std() * len(bimodal)**(-1/5)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].hist(bimodal, bins=50, density=True, color="tab:blue", alpha=0.7)
axes[0].set_title("histogram: two clusters visible")
axes[1].boxplot(bimodal, vert=False, widths=0.5)
axes[1].set_title("box plot: clusters INVISIBLE")
axes[1].set_yticks([])
axes[2].plot(grid, kde(grid, bimodal, sigma_b), color="tab:red", linewidth=2)
axes[2].set_title("KDE: two clear bumps")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("value")
plt.tight_layout()
plt.show()

print("Exercise 4: the histogram and the KDE both reveal the two modes; the box")
print("plot compresses them into one innocent-looking box.")

In [ ]:
# Exercise 5: KDE variance = sample variance + bandwidth^2
grid = np.linspace(-250, 350, 40001)
s2 = newspaper.var()
print(f"Exercise 5: sample variance s^2 = {s2:.2f}")
for s in [2, 6, 25]:
    est = kde(grid, newspaper, s)
    m = np.trapezoid(grid * est, grid)
    v = np.trapezoid((grid - m)**2 * est, grid)
    print(f"  sigma = {s:>2}: KDE variance = {v:>8.2f}   s^2 + sigma^2 = {s2 + s**2:>8.2f}")
print("Smearing each point with an independent N(0, sigma^2) kernel adds sigma^2")
print("to the variance -- the variance-of-a-sum rule in action.")

In [ ]:
# Exercise 6: sampling from the KDE
rng = np.random.default_rng(0)

n_samples = 100_000
chosen = rng.choice(newspaper, size=n_samples)          # step 1: pick a data point
samples = chosen + rng.normal(0, 6, size=n_samples)     # step 2: add kernel noise

grid = np.linspace(-20, 110, 800)
plt.figure(figsize=(9, 4.5))
plt.hist(samples, bins=100, density=True, color="tab:blue", alpha=0.5,
         label="samples from the two-step recipe")
plt.plot(grid, kde(grid, newspaper, 6), color="tab:red", linewidth=2.5,
         label="KDE curve (sigma = 6)")
plt.title("Sampling a data point, then adding kernel noise, draws from the KDE")
plt.xlabel("value")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("Exercise 6: the KDE is an equal-weight MIXTURE of n Gaussians. Sampling a")
print("mixture is exactly: choose a component uniformly (a data point), then draw")
print("from that component (add its Gaussian noise).")

## 8. Conclusion

- A normalized **histogram** is technically a density (positive, area 1), but its bars and spurious peaks are artifacts of binning, not features of the data.
- **Kernel density estimation** builds a smooth estimate: a Gaussian kernel ("little mountain") on each data point, summed and divided by $n$: $\hat f(x) = \tfrac1n \sum_i \varphi(x; x_i, \sigma)$.
- The KDE is a genuine PDF by construction — averaging unit-area curves keeps the area at exactly 1.
- The **bandwidth** $\sigma$ sets how far each point's influence spreads: too small memorizes the sample, too large blurs the structure; rules of thumb like Silverman's give a sensible default.
- Twelve points make a lumpy estimate; thousands make a smooth curve that genuinely resembles the underlying PDF — and unlike a box plot, a KDE shows multimodality.
- Bonus structure: the KDE is a mixture of Gaussians — its mean equals the sample mean, its variance is $s^2 + \sigma^2$, and sampling from it is "pick a point, add noise."

This completes the toolkit for describing and visualizing one distribution. Next, the course moves to **two variables at once**: joint distributions and friends.